<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [13]:
!pip install -q google-generativeai

In [14]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [15]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [16]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [17]:
# 從 Colab Secrets 中獲取 API 金鑰，並使用 .strip() 去除可能存在的換行符號
api_key = userdata.get('gemini').strip()

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
try:
    response = client.models.generate_content(
        model = MODEL_ID, contents="Explain how AI works in a few words"
    )
    print(response.text)
except Exception as e:
    print(f"測試 AI 模型時發生錯誤：{e}")

AI learns patterns from data to make intelligent decisions or predictions.


### 定義 AI 摘要函式

In [18]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [19]:
from datetime import datetime

# 負責將成績寫入 Google Sheet
def save_scores_to_sheet(grade_data):
    global _gc, _ws
    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            # 失敗時維持原狀，不重設清單
            return "錯誤：Google Sheet 未能成功連線。", grade_data, grade_data

    if not grade_data:
        return "提示：清單內沒有資料可供儲存。", grade_data, grade_data

    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')

    for subject, grade_str in grade_data:
        try:
            grade = float(grade_str)
            if 0 <= grade <= 100:
                new_grades_for_sheet.append([today, subject, grade])
            else:
                return f"錯誤：科目「{subject}」的分數不合理。", grade_data, grade_data
        except ValueError:
            return f"錯誤：科目「{subject}」的成績格式錯誤。", grade_data, grade_data

    try:
        _ws.append_rows(new_grades_for_sheet)
        return "成功：成績已寫入 Google Sheet，暫存清單已重設。", [], []
    except Exception as e:
        return f"失敗：寫入試算表時發生錯誤 {e}", grade_data, grade_data

# 負責產生 AI 摘要 (並可選擇是否寫回 Sheet)
def generate_ai_summary_only(grade_data):
    if not grade_data:
        return "請先輸入並加入成績清單，才能產生摘要。"

    # 格式化資料給 AI
    formatted_grades = []
    today = datetime.now().strftime('%Y-%m-%d')
    for subject, grade in grade_data:
        formatted_grades.append([today, subject, grade])

    # 呼叫你原本定義好的 get_ai_summary 函式
    summary = get_ai_summary(formatted_grades)

    # (可選) 如果你也想把摘要存回試算表，可以在這裡保留寫入邏輯
    try:
        next_row = len(_ws.col_values(1)) + 1
        _ws.update_cell(next_row, 1, today)
        _ws.update_cell(next_row, 2, 'AI 摘要')
        _ws.update_cell(next_row, 3, summary)
    except:
        pass # 即使寫入摘要失敗，仍回傳摘要文字給介面

    return summary

In [20]:
# 確保 Google Sheet 連線已經建立或重新建立
setup_gspread(SHEET_URL, WORKSHEET_NAME)

# 準備測試資料
test_grade_data = [
    ["國文", "85"],
    ["數學", "78"],
    ["英文", "92"]
]

print("\n--- 正在執行 process_grades_and_summary 函式單元測試... ---")

sheet_status, ai_summary_output = process_grades_and_summary(test_grade_data)

print("\n--- 函式執行結果 --- ")
print(f"Google Sheet 處理狀態: {sheet_status}")
print(f"AI 摘要:\n{ai_summary_output}")

# 檢查 _ws 是否為 None，判斷 Google Sheet 是否真的連線成功
if _ws is None:
    print("\n注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能仍有問題。")
else:
    print("\nGoogle Sheet 工作表物件 (_ws) 已成功初始化，連線似乎已建立。")

--- 正在進行 Google Sheet 身份驗證和連線... ---
--- Google Sheet 連線成功。---

--- 正在執行 process_grades_and_summary 函式單元測試... ---

--- 正在呼叫 AI 模型生成摘要... ---

--- 函式執行結果 --- 
Google Sheet 處理狀態: 成績已成功寫入 Google Sheet。
AI 摘要已成功寫入 Google Sheet。
AI 摘要:
好的，這是一份根據您提供的學生三科成績所做的簡單摘要與常見迷思整理，全程不對學生的表現進行評分，僅做客觀總結與迷思澄清。

---

### 學生成績摘要

這是一份學生在 **2026年4月19日** 的三科測驗成績列表，包含國文、數學及英文。

*   **國文：85分**
*   **數學：78分**
*   **英文：92分**

從這些數據來看：
*   該學生在這次測驗中，**英文科目得分最高（92分）**。
*   **數學科目得分相對最低（78分）**。
*   **國文科目得分居中（85分）**。
*   所有科目的分數都位於**78分至92分之間**。

---

### 成績解讀常見迷思整理

在解讀學生的成績時，我們常常會不自覺地陷入一些迷思，這些迷思可能導致我們對學生做出不完全或不恰當的判斷。以下是一些常見的迷思及其澄清：

1.  **迷思一：單次成績代表學生的全部能力或努力程度。**
    *   **澄清：** 成績是學生在**特定時間、特定測驗範圍**內的表現。它可能受到多種因素影響，例如當天的身心狀態、試題的難易度、出題方向、甚至是答題策略等。單次的成績並不能完全代表學生的學習潛力、知識掌握的全面性或其在學習上投入的努力。

2.  **迷思二：高分科目代表學生對該科目充滿興趣或天賦，低分科目則相反。**
    *   **澄清：** 成績高低與興趣或天賦不必然畫上等號。高分可能來自於學生有效的學習方法、紮實的練習或對某個特定單元的良好掌握。低分也可能只是對某個概念的不理解、一時粗心或該單元剛好是弱點。興趣和天賦是多面向的，不只體現在分數上。

3.  **迷思三：數學成績較低就代表數學不好或不適合讀理工。**
    *   **澄清：** 針對此列表中的數學成績相對較低

定義 Gradio 處理函式

In [21]:
import gradio as gr

SUBJECTS = ["國文", "英文", "數學", "自然", "社會"]

with gr.Blocks() as demo:
    gr.Markdown("# 成績管理系統 (分步處理版)")

    # 使用 State 儲存目前的成績清單
    grade_state = gr.State([])

    with gr.Row():
        # 左側：輸入與清單管理
        with gr.Column():
            gr.Markdown("### 1. 輸入成績")
            with gr.Row():
                sub_input = gr.Dropdown(choices=SUBJECTS, label="選擇科目", value="國文")
                num_input = gr.Number(label="成績 (0-100)", minimum=0, maximum=100, value=0)

            add_btn = gr.Button("加入預備清單")

            gr.Markdown("### 2. 確認預備清單")
            display_df = gr.Dataframe(
                headers=["科目", "成績"],
                type="array",
                interactive=False
            )
            clear_btn = gr.Button("清空清單", size="sm")

        # 右側：執行動作與結果顯示
        with gr.Column():
            gr.Markdown("### 3. 執行動作")
            with gr.Row():
                save_btn = gr.Button("儲存至試算表", variant="primary")
                ai_btn = gr.Button("產生 AI 摘要", variant="secondary")

            gr.Markdown("### 執行結果")
            status_output = gr.Textbox(label="系統訊息")
            summary_output = gr.Textbox(label="Gemini 分析報告", lines=12)

    # --- 互動邏輯 ---

    # 加入清單
    def add_logic(current_list, sub, score):
        if score is None or score < 0 or score > 100:
            raise gr.Error("請輸入 0 到 100 之間的有效分數")
        updated = current_list + [[sub, score]]
        return updated, updated

    add_btn.click(add_logic, [grade_state, sub_input, num_input], [grade_state, display_df])

    # 清空清單
    clear_btn.click(lambda: ([], []), None, [grade_state, display_df])

    # 按鈕 1：僅儲存
    save_btn.click(
        fn=save_scores_to_sheet,
        inputs=grade_state,
        outputs=status_output
    )

    # 按鈕 2：僅產生摘要
    ai_btn.click(
        fn=generate_ai_summary_only,
        inputs=grade_state,
        outputs=summary_output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7b6d6ed616f147de12.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
